# Cruce con datos públicos reales (INEI) — AlertaSegura Perú

Extra (opcional) del rol de Data Analysis.

Cruza los reportes sintéticos con la **población real por distrito**
(INEI) para calcular una tasa de reportes por cada 10,000 habitantes.
Esto es más justo que solo contar reportes crudos: un distrito grande
como San Juan de Lurigancho va a tener más reportes solo por tener más
gente, no necesariamente porque sea más riesgoso.

**Importante sobre los datos de población**: son cifras reales del INEI
(o estimadas a partir de porcentajes publicados por el INEI), pero de
años distintos según qué se encontró disponible (2015-2024, ver columna
`anio_dato` y `fuente` en `data/poblacion_distritos_lima.csv`). Para un
análisis de producción real convendría homologar todo a un solo año
(el censo 2017 o una proyección INEI 2024 completa), pero para esta
demo del Sprint 4 sirve para ilustrar el método.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

reportes = pd.read_csv("../data/reportes_sinteticos.csv")
poblacion = pd.read_csv("../data/poblacion_distritos_lima.csv")
poblacion.head()

In [ ]:
conteo = reportes["distrito"].value_counts().rename("n_reportes").reset_index()
conteo.columns = ["distrito", "n_reportes"]

cruce = conteo.merge(poblacion, on="distrito", how="left")
cruce["tasa_por_10k_hab"] = (cruce["n_reportes"] / cruce["poblacion"]) * 10_000
cruce = cruce.sort_values("tasa_por_10k_hab", ascending=False)
cruce

## Reportes crudos vs. tasa per cápita

El ranking cambia bastante: un distrito puede verse "tranquilo" en el
conteo crudo simplemente por tener poca gente, y aparecer con más riesgo
relativo al normalizar por población (o al revés).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

orden_crudo = cruce.sort_values("n_reportes")
axes[0].barh(orden_crudo["distrito"], orden_crudo["n_reportes"], color=sns.color_palette("crest", len(orden_crudo)))
axes[0].set_title("Reportes crudos por distrito")
axes[0].set_xlabel("N° de reportes")

orden_tasa = cruce.sort_values("tasa_por_10k_hab")
axes[1].barh(orden_tasa["distrito"], orden_tasa["tasa_por_10k_hab"], color=sns.color_palette("rocket", len(orden_tasa)))
axes[1].set_title("Reportes por cada 10,000 habitantes")
axes[1].set_xlabel("Tasa por 10k hab.")

plt.tight_layout()
plt.show()

In [ ]:
top_crudo = cruce.sort_values("n_reportes", ascending=False).head(5)["distrito"].tolist()
top_tasa = cruce.sort_values("tasa_por_10k_hab", ascending=False).head(5)["distrito"].tolist()

print("Top 5 por reportes crudos:", top_crudo)
print("Top 5 por tasa per cápita:", top_tasa)
print("Distritos que cambian de posición:", set(top_crudo) ^ set(top_tasa))

In [ ]:
cruce.to_csv("../data/reportes_vs_poblacion.csv", index=False)
print("Guardado: reportes_vs_poblacion.csv")

## Fuente de los datos de población

INEI (Instituto Nacional de Estadística e Informática), a partir de
notas de prensa públicas sobre proyecciones poblacionales de Lima
Metropolitana y el Callao (aniversario de Lima 2024, censo 2017, y
proyecciones distritales). Detalle año por año y fuente exacta en
`data/poblacion_distritos_lima.csv`. No se usó la Plataforma Nacional de
Datos Abiertos directamente porque no tiene un dataset único y limpio de
población por distrito descargable en CSV; se optó por cifras de prensa
del propio INEI, que son la fuente primaria de esos artículos.